In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.mixed_precision import set_global_policy
from docx import Document

# Enable mixed precision for faster computation
set_global_policy('mixed_float16')

# Ensure reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Dataset directories
train_dir = 'D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\DataSet\\Train'
val_dir = 'D:\\PROJECT\\Monkeypox Detection Using Machine Learning\\DataSet\\Val'

# Data augmentation
img_height, img_width = 224, 224
batch_size = 64  # Larger batch size for faster training

datagen_train = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, rotation_range=20, zoom_range=0.2, width_shift_range=0.2, height_shift_range=0.2
)
datagen_val = ImageDataGenerator(rescale=1./255)

train_data = datagen_train.flow_from_directory(
    train_dir, target_size=(img_height, img_width), batch_size=batch_size, class_mode='binary'
)
val_data = datagen_val.flow_from_directory(
    val_dir, target_size=(img_height, img_width), batch_size=batch_size, class_mode='binary', shuffle=False
)

# Function to save and display confusion matrix
def save_confusion_matrix(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix for {model_name}:\n{cm}")

    results_dir = 'D:\\PROJECT\\PROGRAM\\RESULTS\\'
    os.makedirs(results_dir, exist_ok=True)

    doc = Document()
    doc.add_heading(f'Confusion Matrix for {model_name}', level=1)
    for row in cm:
        doc.add_paragraph(' '.join(map(str, row)))
    doc.save(os.path.join(results_dir, f'{model_name}_confusion_matrix.docx'))

# Function to train and evaluate models
def train_and_evaluate(model, model_name):
    checkpoint_path = f'D:\\PROJECT\\PROGRAM\\MODELS\\{model_name}.keras'
    checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max')
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    print(f"\nTraining {model_name}...\n")
    history = model.fit(
        train_data,
        epochs=10,  # Reduced epochs
        validation_data=val_data,
        callbacks=[checkpoint, early_stopping]
    )

    best_model = tf.keras.models.load_model(checkpoint_path)
    val_loss, val_accuracy = best_model.evaluate(val_data)
    print(f"{model_name} Validation Accuracy: {val_accuracy * 100:.2f}%")

    y_true = val_data.classes
    y_pred = (best_model.predict(val_data) > 0.5).astype('int32').flatten()

    save_confusion_matrix(model_name, y_true, y_pred)
    return val_accuracy

# ResNet50 Model
resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
for layer in resnet_base.layers[:140]:  # Freeze most layers for faster training
    layer.trainable = False

resnet_model = Model(inputs=resnet_base.input, outputs=Dense(1, activation='sigmoid')(Flatten()(resnet_base.output)))
resnet_model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
resnet_accuracy = train_and_evaluate(resnet_model, 'ResNet50')

# Print Results
print("\nModel Accuracies:")
print(f"ResNet50 Accuracy: {resnet_accuracy * 100:.2f}%")


Found 9900 images belonging to 2 classes.
Found 825 images belonging to 2 classes.

Training ResNet50...



C:\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
  6/155 ━━━━━━━━━━━━━━━━━━━━ 2:52:00 69s/step - accuracy: 0.4636 - loss: 2.5513